# Day Trader Controls

In its nascent stage, the goal of this trading bot is learning and testing rigging. Fees, day-trade rules, and spreads are key concerns. Its architecture design is mostly conservative, focusing on edge validation in out-of-sample data, and testing operational pipelines end to end. This means monitoring the model to avoid chasing small returns that are eaten by market fees. Test results from `day-metrics.ipnyb` adds to this concern, as inital runs yield limited edge in out-of-sample 

This design follows a selective, high-conviction swing strategy plan. This is not a scalper. A trade held for a clean three or five percent move does not care about a 0.2 percent round trip; a trade chasing half a percent is eaten by it, this is our floor.

The minimum-edge floor is enforced to manage high-volume trading where a model of cumulative return appears busier by taking many small trades, while fees quietly eat the account. We propose three guardrails against this: i) a cap limit on number of trades-per-day, ii) minimum-edge floor limit on thinner trades, iii) and after-fee validation protocol. Effectively, this restricts the model from high frequency trades on thinner margins with fees reported. 


### 1. Key Variables

Every tunable parameter below is stored in one `config.py` helper never hard-coded, and the risk-bounding
ones are operator-owned and held outside the model's reach.

**Selection screen (per-coin, computed before the model sees the coin)**
- `quote_volume_24h_usdt` — 24h quote volume, USDT. Hard gate. 🔒 floor
- `atr_pct` — ATR(14) ÷ current price, as a percent. Band, two edges. 🔒 floor and ceiling
- `spread_pct` — (ask − bid) ÷ mid, as a percent. Hard gate. 🔒 ceiling
- `candle_count` — number of daily candles returned. Sufficiency gate. 🔒 minimum
- `pass` — boolean, true only if all gates clear

**Edge and exit (per-trade economics)**
- `round_trip_fee_pct` — venue round-trip cost (Binance ≈ 0.15–0.20%)
- `expected_slippage_pct` — modelled slippage per round trip
- `edge_floor_pct` — minimum net expected move to allow a trade. 🔒
- `take_profit_pct` — target gain; must exceed `edge_floor_pct`
- `stop_loss_pct` — max loss per trade before forced exit
- `net_expected_move = est_move_pct − round_trip_fee_pct − expected_slippage_pct`

**Position and frequency limits**
- `max_trades_per_day` — hard cap. 🔒
- `max_open_positions` — concurrent positions (3–4 at current account size). 🔒
- `position_size_pct` — capital per trade; volatility-scaled (see §3)
- `hold_window_days` — range, fitted by walk-forward, not fixed (swing band)

**Signal layer (from existing day-metrics notebook)**
- `macd`, `macd_signal`, `macd_hist`, `hist_slope`, `converging`
- `cross_up`, `cross_down`, `guarded_buy`, `guarded_sell`, `epsilon` (noise band)
- `bear_div`, `bull_div` (swing-pivot divergence flags)
- four-vote score: `w_macd*MACD + w_ma*MA + w_fib*Fib + w_candle*Candle`
- `threshold` — vote score to fire (2 standard, 3 when few trades to learn from)

**Evaluation harness (frozen, versioned, operator-owned)** 🔒
- `oos_window`, `train_window`, `regime_set`, `fee_assumption` — all fixed across comparisons
- `experiment_log` — one line per walk-forward run: params, OOS after-fee result, kept/discarded


### The three stacked defences against volume-hiding

- **Who sets the floor.** The minimum-edge floor is set by the operator or fixed outside the model's reach, never tuned by the model itself. The model is judged on the results the floor produces, so a model that could lower the floor would lower it to book more wins. The floor is reviewed weekly by the operator, not learned.
- **Defence one — trades-per-day cap.** A hard limit on how many trades can exist in a day, set in code. This alone makes the volume trick mechanically impossible: the model cannot churn because it cannot place the orders.
- **Defence two — minimum-edge floor.** A trade is refused unless its expected net move clears the floor. Net means after round-trip fee and after expected slippage, not gross. On Binance crypto the round trip is roughly 0.2 percent, so the floor sits well above that, in the region of one and a half to two percent expected move.
- **Defence three — out-of-sample validation.** Walk-forward only. Tune weights and threshold on a training segment, score once on an untouched test segment, move forward, repeat. Report only the out-of-sample, after-fee, multi-regime aggregate. No in-sample result is ever the verdict.

### The floor is a fence, not an alarm

- The floor **refuses** the trade. It is not a flag the model can ring and then trade anyway. An alarm that lets the trade through does not stop the loss.
- Keep an optional **alarm band** just above the fence as an early warning, but the fence below it is what protects the account.
- The floor is measured on **estimated move minus round-trip fee minus expected slippage**. That is the honest waterline, and it is higher than the raw fee.

### How the floor relates to the exit

- The take-profit target must sit **above** the floor. A take-profit below the floor is incoherent: aiming to exit at a gain the entry rule calls too thin to take.
- Floor and take-profit are **set together**, not independently.
- Set too low, the floor lets churn through. Set too high, the model sits on its hands and never trades, which can look like success right up until you notice no trades cleared in a month. The floor is a tuned parameter, reviewed weekly.

### Two venues, two fence sets

- **Binance crypto.** Round-trip cost roughly 0.15 to 0.2 percent (0.075 percent per side paying fees in BNB, 0.1 percent otherwise). Fee is the binding fence. Keep a BNB balance topped up for the discount.
- **Alpaca equities.** Per-trade cost effectively zero (regulatory pass-throughs on sells only, cents per trade). The binding fence is the pattern-day-trader rule: a sub-25k cash account is capped at three day-trades per rolling five days. Margin interest at 6.25 percent annualized applies to any leveraged overnight hold — trade cash-only to avoid it. Confirm the account is direct self-directed, not partner-routed, or the zero-commission assumption breaks.
- The model must know **which fence set governs the order it is about to place.**

### Volatility Filter 

- ATR Band: Average true range across a 14 day period reported as percentage of current price. ATR indicator of typical daily movement.
- Floor: a coin must move enough per day to reach the take-profit inside the hold window. Floor sits above the net edge requirement (i.e. ~2-3% daily ATR).
- Ceiling: above some ATR the coin gaps through stop and take-profit unpredictably and indicators lose meaning. Ceiling sits below where the coin detonates.
- These inform two unrelated operations:
    1. Selection Filter: the gate that admits or rejects a coin from the universe before the model sees it.
    2. Live Guardrail: keeps the model from trading a coin that has drifted out of the tradable band.



### 2. Four-Gate Selection Rule

We propose the following screening assessment to evaluate coins individually on a weekly basis.

Inputs: daily OHLCV, last 90–180 candles, plus current order-book top.

- Gate 1 Liquidity - used to reject below floor before computing anything else.
 - `quote_volume_24h_usdt` from the ticker. 
 
- Gate 2 ATR Band - reported as percent of price showing average true range per day using:  
  - `TR = max(high−low, |high−prev_close|, |low−prev_close|)`
  - `14-period moving average of TR`
  - `atr_pct = ATR / current_price × 100`
  - `atr_floor_pct ≤ atr_pct ≤ atr_ceiling_pct`. 

- Gate 3 Spread - Rejected if above a tight ceiling that signficantly under the fee floor, derived from order book as:
  - `spread_pct = (ask − bid) / mid × 100` 

- Gate 4 History Range -  Rejects young tokens where Ichimoku cloud clearly lacking:
  - `candle_count < min_history`. 

This informs selection based on `{symbol, quote_volume_24h_usdt, atr_pct, spread_pct, candle_count, pass}`.
Targets include a sample of 15–25 coins allowing positions on 3–4 coins.



### 3. Edge floor, exit, and position sizing (the quantitative core)

**Net edge test (entry gate).** A trade is refused unless:
`net_expected_move = est_move_pct − round_trip_fee_pct − expected_slippage_pct ≥ edge_floor_pct`
Measured on **net**, not gross. This is the fence, not an alarm — it refuses, it does not warn.

**Exit coupling.** Required ordering, enforce in code:
`edge_floor_pct < take_profit_pct` and `atr_floor_pct` large enough that a normal multi-day range
reaches `take_profit_pct` inside `hold_window_days`. `stop_loss_pct` set against the same ATR.

**Volatility-scaled position sizing (industry standard, ATR-based).** Keep dollar risk roughly
constant across coins: size each position so that `stop_loss_pct × position_value ≈ constant risk
budget`. Higher-ATR coin → smaller position; calmer coin → larger. At ~$500–1000 account, floor
position size at the venue's minimum notional, and prefer 3–4 larger positions over many tiny clips
that waste edge on fees and bump minimums.

---

### 4. Hold period — resolved as a range, not a fixed number

The earlier "1–3 days" was a gut figure. Resolution: **swing is the band** (days to ~2 weeks),
scalping and day-trading ruled out (fees and PDT rule), position-trading ruled out (underuses the
signal stack). Within the swing band, make `hold_window_days` a **walk-forward parameter** (e.g.
search 1–10 days) and let out-of-sample results pick it. Shorter holds demand higher `atr_pct`;
longer holds tolerate lower. Same dial, two ends.

---

### 5. Autoresearch lifts (from karpathy/autoresearch, adapted)

The repo's architecture maps onto this project, but its separation is enforced by **instruction**
("do not modify this file"); a model optimised against the metric needs separation enforced by
**code structure**. Lifts:
- **Narrow editable surface.** Define the one file/module the model may tune; lock everything else
  (evaluation, fee floor, ATR band, guardrails) in files the model cannot write to.
- **Experiment log.** Each walk-forward run writes one line: params tried, OOS after-fee result,
  kept or discarded, why. This is the artefact the operator reviews weekly.
- **Frozen yardstick.** Same OOS windows, fees, regime set across every comparison. Any change to
  the harness is a deliberate, separately-recorded human act, never mid-comparison.
- **program.md pattern.** The human-owned instruction layer (this document / PROJECT-BRIEF.md) is
  iterated by the operator; the model iterates the strategy. Mirrors his program.md vs train.py split.
- **Caution:** trading is adversarial and non-stationary; an OOS metric can rot live in a way his
  fixed-corpus val_bpb never does. The weekly human review is not automatable away.

---

### 6. Design parameters (carried forward, unchanged)

#### Three stacked defences against volume-hiding
- **Who sets the floor.** Operator-set or fixed outside the model's reach, never tuned by the model. Reviewed weekly.
- **Defence one — trades-per-day cap.** Hard, in code. Churn becomes mechanically impossible.
- **Defence two — minimum-edge floor.** Refuses trades whose net move does not clear the floor.
- **Defence three — out-of-sample validation.** Walk-forward only; report OOS after-fee multi-regime aggregate.

#### Floor is a fence, not an alarm
- Refuses the trade; optional alarm band may sit above the fence, but the fence protects the account.
- Measured on estimated move minus fee minus slippage — the honest waterline, above the raw fee.

#### Two venues, two fence sets
- **Binance crypto.** Round trip ≈ 0.15–0.20% (0.075%/side with BNB). Fee is the binding fence. Keep BNB topped up.
- **Alpaca equities.** Per-trade ≈ zero (reg pass-throughs, sells only). Binding fence is the PDT rule: sub-25k cash account capped at 3 day-trades / rolling 5 days. Margin interest 6.25%/yr on leveraged overnight holds — trade cash-only. Confirm account is direct self-directed, not partner-routed (else 0%-3% commission band applies).
- Model must know which fence set governs each order.